In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

In [7]:
load_dotenv()

True

In [8]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

In [13]:
DATA_FILE_PATH = os.path.join(".." ,"data", "hr_policy.txt")

## DATA INGESTION

In [14]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': '..\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring pro

In [15]:
len(documents)

1

In [16]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [18]:
print(documents[0].metadata)

{'source': '..\\data\\hr_policy.txt'}


In [19]:
print(len(documents[0].page_content))

2597


## SPLITTING OUR DATA

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)
print(chunks)

# Now each chunk is a document in itself

[Document(metadata={'source': '..\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': '..\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': '..\\data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core 

In [21]:
len(chunks)

9

## Embeddings

In [24]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### Store data in vector db

In [25]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

print("Chunks are stored", vector_store.index.ntotal)

Chunks are stored 9


In [26]:
test_query = "How many sickk leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query: {test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"---Match {i} ---")
    print(match.page_content)
    print()

Query: How many sickk leaves employees get

---Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### Tool

In [31]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(question: str) -> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)


### Data Retrieval

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model= "openai/gpt-oss-120b",
    temperature=0
)

llm.model_name

'openai/gpt-oss-120b'

In [33]:
test_response = llm.invoke("Hey is learning rag hard? answer in 1 line")
print(test_response.content)

Learning RAG can be challenging at first, but with the right resources and practice, it becomes manageable.


## AI AGENT

In [34]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt= """

    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."

    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [35]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" *  60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    # response gives system msg, then human msg and then ai msg thats why we do [-1]
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)

In [36]:

response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [37]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='a1a8ec8b-6c37-45a9-a8a9-7395dbcd3fdc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Must answer using search_hr_policy tool to look up facts before answering. The question is about which organization the assistant works for. This is presumably known: Acme Crop. But the instruction says always use the search_hr_policy tool to look up facts before answering. The policy document likely contains organization name? Possibly not. But we can still search. Let\'s search for "Acme Crop".', 'tool_calls': [{'id': 'fc_ca2cc017-e496-48f4-99c6-2bcd0c6a3e83', 'function': {'arguments': '{"question":"Acme Crop organization name"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_token

In [40]:
print(response)

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='a1a8ec8b-6c37-45a9-a8a9-7395dbcd3fdc'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Must answer using search_hr_policy tool to look up facts before answering. The question is about which organization the assistant works for. This is presumably known: Acme Crop. But the instruction says always use the search_hr_policy tool to look up facts before answering. The policy document likely contains organization name? Possibly not. But we can still search. Let\'s search for "Acme Crop".', 'tool_calls': [{'id': 'fc_ca2cc017-e496-48f4-99c6-2bcd0c6a3e83', 'function': {'arguments': '{"question":"Acme Crop organization name"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens'

In [41]:
print(response["messages"][-1].content)

I’m part of the HR team at **Acme Corp** (the organization referred to as Acme Crop in our internal communications).
